# Notebook 1 - Data Loading and Cleaning

## Objective

This notebook loads the six Fama–French datasets used throughout the project, performs initial cleaning, converts returns from percentages to decimal form, aligns portfolio returns with the Fama–French factor datasets, and saves standardized datasets for downstream feature engineering and model training.

### Input datasets

- 25 Portfolios (5×5) Monthly
- 25 Portfolios (5×5) Daily
- Fama–French 3 Factors Monthly
- Fama–French 3 Factors Daily
- Fama–French 5 Factors Monthly
- Fama–French 5 Factors Daily

### Outputs

- Cleaned monthly and daily datasets
- Aligned portfolio + factor datasets

We will be building an experiment around changing at least two of the following factors. We want to better understand how ML understands financial data where information is limited and chaotic.

Frequency:
1. Monthly
2. Daily

Factor model:
1. FF3
2. FF5

Feature type:
1. Signature
2. GARCH
3. Signature + GARCH

Model:
1. CNN
2. XGBoost
3. Ensemble

In [102]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DIR.mkdir(exist_ok=True)

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

FILES = {
    "monthly": {
        "portfolios": RAW_DIR / "25_Portfolios_5x5.csv",
        "ff3": RAW_DIR / "F-F_Research_Data_Factors.csv",
        "ff5": RAW_DIR / "F-F_Research_Data_5_Factors_2x3.csv",
    },

    "daily": {
        "portfolios": RAW_DIR / "25_Portfolios_5x5_Daily.csv",
        "ff3": RAW_DIR / "F-F_Research_Data_Factors_daily.csv",
        "ff5": RAW_DIR / "F-F_Research_Data_5_Factors_2x3_daily.csv",
    }
}

In [104]:
##
with open(FILES["portfolios_monthly"], "r") as f:
    lines = f.readlines()
useful = lines[1:1200]
with open("../data/25_Portfolios_5x5_Usable_Monthly.csv", "w") as f:
    f.writelines(useful)

## 
with open(FILES["portfolios_daily"], "r") as f:
    lines = f.readlines()
useful = lines[1:26254]
with open("../data/25_Portfolios_5x5_Usable_Daily.csv", "w") as f:
    f.writelines(useful)

##
with open(FILES["ff3_monthly"], "r") as f:
    lines = f.readlines()
useful = lines[:1200]
with open("../data/F-F-3_Research_Usable_Monthly.csv", "w") as f:
    f.writelines(useful)

##
with open(FILES["ff3_daily"], "r") as f:
    lines = f.readlines()
useful = lines[:]
with open("../data/F-F-3_Research_Usable_Daily.csv", "w") as f:
    f.writelines(useful)

##
with open(FILES["ff5_monthly"], "r") as f:
    lines = f.readlines()
useful = lines[:756]
with open("../data/F-F-5_Research_Usable_Monthly.csv", "w") as f:
    f.writelines(useful)

##
with open(FILES["ff5_daily"], "r") as f:
    lines = f.readlines()
useful = lines[:]
with open("../data/F-F-5_Research_Usable_Daily.csv", "w") as f:
    f.writelines(useful)

In [105]:
# Verify folders
print("=" * 60)
print("Project Directories")
print("=" * 60)

print(f"Data folder exists:      {DATA_DIR.exists()}")
print(f"Processed folder exists: {PROCESSED_DIR.exists()}")
print()

# Verify each dataset
print("=" * 60)
print("Dataset Verification")
print("=" * 60)

for name, path in FILES.items():

    if path.exists():
        size_mb = path.stat().st_size / (1024**2)

        print(f"✓ {name}")
        print(f"   Location : {path}")
        print(f"   Size     : {size_mb:.2f} MB\n")

    else:
        print(f"✗ {name} NOT FOUND")
        print(f"   Expected : {path}\n")

Project Directories
Data folder exists:      True
Processed folder exists: True

Dataset Verification
✓ portfolios_monthly
   Location : ..\data\raw\25_Portfolios_5x5.csv
   Size     : 2.22 MB

✓ portfolios_daily
   Location : ..\data\raw\25_Portfolios_5x5_Daily.csv
   Size     : 22.91 MB

✓ ff3_monthly
   Location : ..\data\raw\F-F_Research_Data_Factors.csv
   Size     : 0.05 MB

✓ ff3_daily
   Location : ..\data\raw\F-F_Research_Data_Factors_daily.csv
   Size     : 1.15 MB

✓ ff5_monthly
   Location : ..\data\raw\F-F_Research_Data_5_Factors_2x3.csv
   Size     : 0.05 MB

✓ ff5_daily
   Location : ..\data\raw\F-F_Research_Data_5_Factors_2x3_daily.csv
   Size     : 0.97 MB



## Usable Dataset

In [106]:
DATA = {
    "portfolios_monthly": DATA_DIR / "25_Portfolios_5x5_Usable_Monthly.csv",
    "portfolios_daily": DATA_DIR / "25_Portfolios_5x5_Usable_Daily.csv",
    "ff3_monthly": DATA_DIR / "F-F-3_Research_Usable_Monthly.csv",
    "ff3_daily": DATA_DIR / "F-F-3_Research_Usable_Daily.csv",
    "ff5_monthly": DATA_DIR / "F-F-5_Research_Usable_Monthly.csv",
    "ff5_daily": DATA_DIR / "F-F-5_Research_Usable_Daily.csv"
}

## Check the Datasets

In [107]:
datasets = {}

for name, path in DATA.items():
    datasets[name] = pd.read_csv(path)

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("=" * 80)

    print(f"Shape: {df.shape}")
    print(f"Date Range: {df['Date'].min()} -> {df['Date'].max()}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nLast 5 rows:")
    display(df.tail())

    print("\nMissing Values:")
    print(df.isna().sum().sum())
    if df.isna().sum().sum() > 0:
        print(df.isna().sum())
    print("\n")

portfolios_monthly
Shape: (1198, 26)
Date Range: 192607 -> 202604

Columns:
['Date', 'SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM', 'ME2 BM1', 'ME2 BM2', 'ME2 BM3', 'ME2 BM4', 'ME2 BM5', 'ME3 BM1', 'ME3 BM2', 'ME3 BM3', 'ME3 BM4', 'ME3 BM5', 'ME4 BM1', 'ME4 BM2', 'ME4 BM3', 'ME4 BM4', 'ME4 BM5', 'BIG LoBM', 'ME5 BM2', 'ME5 BM3', 'ME5 BM4', 'BIG HiBM']

First 5 rows:


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
0,192607,5.8276,-1.7006,0.5118,-2.1477,1.9583,1.2118,2.4107,0.6056,-2.6082,...,1.5376,1.5460,1.3389,0.2765,2.4678,3.3248,6.0909,2.0285,3.1263,0.5623
1,192608,-2.0206,-8.0282,1.3968,2.1483,8.5104,2.3620,-0.7525,3.8984,0.2299,...,1.3858,3.8587,1.9738,2.1336,5.3422,1.0169,4.1975,1.9769,5.4924,7.7576
2,192609,-4.8291,-2.6806,-4.3417,-3.2683,0.8586,-2.6849,-0.5252,1.0789,-3.2877,...,1.6897,-0.5246,-1.7724,1.4806,0.8730,-1.2951,3.6610,0.1384,-0.7497,-2.4284
3,192610,-9.3633,-3.5519,-3.5024,3.4413,-2.5452,-2.8014,-4.4191,-5.0767,-8.0271,...,-3.9136,-2.6528,-2.1058,-3.2532,-5.3525,-2.7382,-3.0061,-2.2467,-4.6725,-5.8129
4,192611,5.5888,4.1877,2.4384,-4.4495,0.5110,3.1023,-1.7317,3.0425,4.9538,...,3.4492,2.3823,3.7315,5.1102,1.8213,4.4331,2.5355,1.5280,3.6596,2.5636



Last 5 rows:


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
1193,202512,0.2815,1.3821,-1.5678,1.7159,0.9990,2.7014,1.0433,0.1052,2.6181,...,0.3479,-0.1126,2.1711,1.1429,3.8270,-0.6739,0.3489,2.8190,1.1900,4.2624
1194,202601,6.6674,0.9116,4.1966,5.0291,5.8913,1.1545,4.8183,6.6810,7.5831,...,0.2710,4.9754,4.4762,6.3860,5.7044,-0.2591,2.1192,6.2308,2.0611,4.1660
1195,202602,-4.0986,2.4008,0.0104,1.2797,2.1030,-0.1138,-0.8251,2.2820,1.8037,...,-1.0654,5.2847,4.1865,3.9867,2.9580,-3.0626,2.4614,2.7392,2.1192,-0.5265
1196,202603,-5.9641,-1.6495,-0.9363,-0.0516,-2.6779,-5.2841,-3.2980,-5.9152,-1.3210,...,-6.1039,-6.6964,-4.7658,-3.1826,-0.8125,-5.4346,-5.3979,-1.8548,-1.3843,-3.9298
1197,202604,14.9088,11.8988,9.8256,6.1535,9.3208,18.2754,15.0323,13.1418,8.3923,...,6.8708,9.0172,10.5475,4.6235,6.3781,12.2099,7.9694,5.9225,1.2836,31.6629



Missing Values:
0


portfolios_daily
Shape: (26252, 26)
Date Range: 19260701 -> 20260528

Columns:
['Date', 'SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM', 'ME2 BM1', 'ME2 BM2', 'ME2 BM3', 'ME2 BM4', 'ME2 BM5', 'ME3 BM1', 'ME3 BM2', 'ME3 BM3', 'ME3 BM4', 'ME3 BM5', 'ME4 BM1', 'ME4 BM2', 'ME4 BM3', 'ME4 BM4', 'ME4 BM5', 'BIG LoBM', 'ME5 BM2', 'ME5 BM3', 'ME5 BM4', 'BIG HiBM']

First 5 rows:


...


Missing Values:
0


ff5_daily
Shape: (15833, 7)
Date Range: 19630701 -> 20260529

Columns:
['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

First 5 rows:


,Date,Mkt-RF,SMB,HML,RMW,CMA,RF
0,19630701,-0.67,0.00,-0.34,-0.01,0.16,0.01
1,19630702,0.79,-0.26,0.26,-0.07,-0.20,0.01
2,19630703,0.63,-0.17,-0.09,0.18,-0.34,0.01
3,19630705,0.40,0.08,-0.27,0.09,-0.34,0.01
4,19630708,-0.63,0.04,-0.18,-0.29,0.14,0.01



Last 5 rows:


,Date,Mkt-RF,SMB,HML,RMW,CMA,RF
15828,20260522,0.44,0.14,0.19,-0.39,0.58,0.02
15829,20260526,0.65,0.84,0.26,-1.32,-0.38,0.02
15830,20260527,0.02,0.37,-0.35,0.07,-0.18,0.02
15831,20260528,0.67,-0.16,-0.59,-1.08,0.05,0.02
15832,20260529,0.19,-1.03,-0.26,-0.66,0.15,0.02



Missing Values:
0


